In [1]:
import os

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/man-bot-detection'

def human_size(num_bytes):
    for unit in ['Б', 'КБ', 'МБ', 'ГБ']:
        if num_bytes < 1024:
            return f"{num_bytes:.1f} {unit}"
        num_bytes /= 1024
    return f"{num_bytes:.1f} ТБ"

def dir_summary(path, max_list=5):
    """Повертає (існує?, кількість файлів, сумарний розмір, приклад файлів)."""
    if not os.path.isdir(path):
        return False, 0, 0, []
    total_size = 0
    file_count = 0
    examples = []
    for root, dirs, files in os.walk(path):
        for f in files:
            fp = os.path.join(root, f)
            try:
                total_size += os.path.getsize(fp)
            except OSError:
                continue
            file_count += 1
            if len(examples) < max_list:
                examples.append(os.path.relpath(fp, path))
    return True, file_count, total_size, examples

print(f"{'='*70}\nПеревірка: {PROJECT_DIR}\n{'='*70}")
if not os.path.isdir(PROJECT_DIR):
    print("Папки man-bot-detection на Drive немає.")
else:
    print("Папка проєкту існує.")

In [2]:
delbot_dir = os.path.join(PROJECT_DIR, 'data/raw/delbot_mouse')
alt_delbot = os.path.join(PROJECT_DIR, 'raw/delbot_mouse')

print(f"\n{'='*70}\nDELBOT-Mouse\n{'='*70}")

exists, count, size, examples = dir_summary(delbot_dir)
if exists:
    circles_dirs = [d for d in os.listdir(delbot_dir)
                     if os.path.isdir(os.path.join(delbot_dir, d)) and d.startswith('circles_')]
    print(f"Шлях {delbot_dir} існує.")
    print(f"   Файлів усього: {count}, розмір: {human_size(size)}")
    print(f"   Папок circles_*: {len(circles_dirs)} -> {sorted(circles_dirs)}")
    if not circles_dirs:
        print("   Папка є, але жодної підпапки circles_* не знайдено — "
              "перевір, чи не лежать вони на рівень глибше (наприклад, у 'python/').")
else:
    print(f"Шляху {delbot_dir} немає — DELBOT-Mouse доведеться качати.")

exists_alt, count_alt, size_alt, _ = dir_summary(alt_delbot)
if exists_alt:
    print(f"\nТакож знайдено дані в альтернативному шляху {alt_delbot} "
          f"({count_alt} файлів, {human_size(size_alt)}). Можеш видалити вручну, "
          f"щоб не плутатись і не займати місце.")

In [3]:
cic_dir = os.path.join(PROJECT_DIR, 'data/raw/cic_ids_2017')
alt_cic = os.path.join(PROJECT_DIR, 'raw/cic_ids_2017')

print(f"\n{'='*70}\nCIC-IDS2017\n{'='*70}")

exists, count, size, examples = dir_summary(cic_dir)
if exists:
    all_csv = [f for f in os.listdir(cic_dir) if f.endswith('.csv')] if os.path.isdir(cic_dir) else []
    print(f"Шлях {cic_dir} існує.")
    print(f"   Файлів усього: {count}, розмір: {human_size(size)}")
    print(f"   CSV-файлів: {len(all_csv)} -> {sorted(all_csv)}")
    zip_leftover = os.path.join(cic_dir, 'MachineLearningCSV.zip')
    if os.path.exists(zip_leftover):
        zsize = os.path.getsize(zip_leftover)
        status = "схоже на справжній архів" if zsize > 100_000_000 else "підозріло малий — ймовірно, зіпсований HTML-редирект, не справжній zip"
        print(f"   Знайдено MachineLearningCSV.zip ({human_size(zsize)}) — {status}")
    if len(all_csv) >= 3:
        print("   Схоже, усі 3 CSV CIC-IDS2017 вже є — завантажувати заново не потрібно.")
    elif all_csv:
        print(f"   Знайдено лише {len(all_csv)} CSV з очікуваних 3 (Friday-Afternoon-DDos, Friday-Morning, Wednesday). Може знадобитись довантажити решту.")
else:
    print(f"Шляху {cic_dir} немає — CIC-IDS2017 доведеться качати.")

exists_alt, count_alt, size_alt, _ = dir_summary(alt_cic)
if exists_alt:
    print(f"\nТакож знайдено дані в альтернативному шляху {alt_cic} "
          f"({count_alt} файлів, {human_size(size_alt)}). Можеш видалити вручну.")

In [4]:
for name, subdir in [
    ('UNSW-NB15', 'data/raw/unsw_nb15'),
    ('CTU-13', 'data/raw/ctu13'),
    ('IoT-23', 'data/raw/iot23'),
]:
    full_path = os.path.join(PROJECT_DIR, subdir)
    print(f"\n{'='*70}\n{name} ({subdir})\n{'='*70}")
    exists, count, size, examples = dir_summary(full_path)
    if not exists:
        print(f"Папки немає — {name} доведеться качати/докладати вручну.")
        continue
    print(f"Папка існує. Файлів усього: {count}, розмір: {human_size(size)}")
    print(f"   Приклади файлів: {examples}")
    if name == 'IoT-23':
        pcap_files = []
        for root, dirs, files in os.walk(full_path):
            pcap_files.extend([f for f in files if f.endswith(('.pcap', '.pcapng'))])
        if not pcap_files:
            print("   pcap-файлів немає — етап 04_network_timeseries.ipynb "
                  "не зможе побудувати часові послідовності.")

In [5]:
hf_cache = os.path.expanduser('~/.cache/huggingface/datasets')
if os.path.isdir(hf_cache):
    print(f"Локальний кеш HuggingFace в цій Colab-сесії існує: {hf_cache}")
else:
    print("Кешу HuggingFace в цій сесії ще немає — CaptchaSolve30k завантажиться при запуску 01.")